# Clasificación ABC-XYZ de inventario

**Objetivo:** clasificar los productos de un comercio minorista según **cuánto aportan a los ingresos (ABC)** y **qué tan estable es su demanda (XYZ)**, para proponer una política de inventario distinta para cada grupo.

**Datos:** [Online Retail II (UCI)](https://archive.ics.uci.edu/dataset/502/online+retail+ii): ~1 millón de transacciones de un comercio online del Reino Unido entre 2009 y 2011. Licencia CC BY 4.0.

## 1. Configuración y descarga de datos

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from download_data import download
from abc_xyz import load_raw, clean, abc_xyz, POLICIES

sns.set_theme(style="whitegrid")
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

xlsx_path = download()  # solo descarga la primera vez

## 2. Carga y limpieza

In [ ]:
raw = load_raw(xlsx_path)
print(f"Filas originales: {len(raw):,}")
raw.head()

In [ ]:
df = clean(raw)
print(f"Filas después de limpiar: {len(df):,} ({len(df) / len(raw):.1%} del total)")
print(f"Productos (SKU): {df['StockCode'].nunique():,}")
print(f"Periodo: {df['InvoiceDate'].min():%Y-%m-%d} a {df['InvoiceDate'].max():%Y-%m-%d}")
print(f"Ingresos totales: £{df['Revenue'].sum():,.0f}")

**Reglas de limpieza:**
- Se eliminan las facturas de cancelación (empiezan por `C`), las cantidades o precios ≤ 0 y los duplicados.
- Se excluyen los códigos que no son productos (portes `POST`, ajustes, comisiones de Amazon, etc.).

## 3. Clasificación ABC-XYZ

- **ABC (valor):** A = SKUs que generan el primer 80 % de los ingresos, B = el siguiente 15 %, C = el 5 % restante.
- **XYZ (variabilidad):** coeficiente de variación (CV) de la demanda **semanal**. X = CV ≤ 0,5 (estable), Y = 0,5–1 (variable), Z = CV > 1 (errática).

In [ ]:
result = abc_xyz(df, freq="W")
result.head(10)

## 4. Curva de Pareto

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = range(1, len(result) + 1)
ax.plot([i / len(result) * 100 for i in x], result["CumShare"] * 100, color="#1f4e79", lw=2)
ax.axhline(80, ls="--", color="grey", lw=1)
ax.axhline(95, ls="--", color="grey", lw=1)
share_a = (result["ABC"] == "A").mean() * 100
ax.axvline(share_a, ls=":", color="#c0392b", lw=1.5)
ax.set_xlabel("% de productos (ordenados por ingresos)")
ax.set_ylabel("% acumulado de ingresos")
ax.set_title(f"Curva de Pareto: el {share_a:.0f} % de los productos genera el 80 % de los ingresos")
fig.tight_layout()
fig.savefig(FIG_DIR / "pareto.png", dpi=150)
plt.show()

## 5. Matriz ABC-XYZ

In [ ]:
order_abc, order_xyz = ["A", "B", "C"], ["X", "Y", "Z"]
count_matrix = pd.crosstab(result["ABC"], result["XYZ"]).reindex(index=order_abc, columns=order_xyz, fill_value=0)
revenue_matrix = (
    result.pivot_table(index="ABC", columns="XYZ", values="RevenueShare", aggfunc="sum", fill_value=0)
    .reindex(index=order_abc, columns=order_xyz, fill_value=0) * 100
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(count_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("Número de productos")
sns.heatmap(revenue_matrix, annot=True, fmt=".1f", cmap="Greens", cbar=False, ax=axes[1])
axes[1].set_title("% de los ingresos")
for ax in axes:
    ax.set_xlabel("Variabilidad de la demanda (XYZ)")
    ax.set_ylabel("Valor (ABC)")
fig.tight_layout()
fig.savefig(FIG_DIR / "abc_xyz_matrix.png", dpi=150)
plt.show()

## 6. Resumen por clase y política de inventario recomendada

In [ ]:
summary = (
    result.groupby("Class")
    .agg(SKUs=("Revenue", "size"), Ingresos=("Revenue", "sum"), CV_medio=("CV", "mean"))
    .assign(PctSKUs=lambda t: t["SKUs"] / t["SKUs"].sum() * 100,
            PctIngresos=lambda t: t["Ingresos"] / t["Ingresos"].sum() * 100)
    .round({"Ingresos": 0, "CV_medio": 2, "PctSKUs": 1, "PctIngresos": 1})
)
summary["Política recomendada"] = summary.index.map(POLICIES)
summary

## 7. Exportar resultados

In [ ]:
out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
result.to_csv(out_dir / "abc_xyz_por_sku.csv")
summary.to_csv(out_dir / "resumen_por_clase.csv")
print("Archivos guardados en data/processed/")

## 8. Conclusiones

> Completa esta sección con los números que obtengas al ejecutar el notebook:

- **Concentración:** ¿qué % de los productos genera el 80 % de los ingresos?
- **Productos AX:** son los más valiosos y predecibles. Conviene automatizar su reposición.
- **Productos AZ:** valiosos pero erráticos. Son el mayor riesgo de roturas de stock y requieren seguimiento cercano con Compras.
- **Productos CZ:** poco valor y demanda errática. Candidatos a descatalogar o a vender solo bajo pedido para liberar capital y espacio.

**Próximos pasos:** calcular el stock de seguridad y el punto de reorden por clase, y validar los resultados con un análisis por año (2010 vs. 2011).